# AnnotationAgent — Demo (Assignment 3)

Этот ноутбук:
- запускает auto_label для текстовой модальности
- генерирует annotation_spec.md
- считает метрики качества
- экспортирует задачи в Label Studio
- создаёт очередь low-confidence для HITL


In [ ]:
from pathlib import Path
import pandas as pd

from annotation_agent import AnnotationAgent

data_dir = Path('data/raw')
candidates = [
    data_dir / 'merged_with_appendix.parquet',
    data_dir / 'merged_raw.parquet',
    data_dir / 'merged_raw.csv',
]
path = next((p for p in candidates if p.exists()), None)
assert path is not None, f'No dataset found. Looked for: {candidates}'
print('Loading:', path)

df = pd.read_parquet(path) if path.suffix == '.parquet' else pd.read_csv(path)
df.head()

In [ ]:
agent = AnnotationAgent(modality='text', config={
    'confidence_threshold': 0.7,
    'include_predictions': True,
})
df_labeled = agent.auto_label(df)
df_labeled[['uid','text','label','confidence','seed_role','label_reason']].head(10)

In [ ]:
spec_path = agent.generate_spec(df_labeled, task='ru_fpbench_borderline_prompt_classification')
print('Spec saved to:', spec_path)

In [ ]:
metrics = agent.check_quality(df_labeled)
metrics

In [ ]:
ls_path = agent.export_to_labelstudio(df_labeled)
print('Label Studio import saved to:', ls_path)
print('Low-conf queue (csv) exists:', Path('data/labeled/review_queue.csv').exists())
print('Low-conf tasks (json) exists:', Path('data/labeled/labelstudio_low_confidence.json').exists())
